In [ ]:
import pandas as pd
import librosa
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

data = pd.read_csv(r'UrbanSound8K.csv')


data.head()

In [ ]:
#Testing if audio path file works
test_audio = r"UrbanSound8k\fold5/100032-3-0-0.wav"
Audio(test_audio)

In [ ]:
#attempt on librosa conversion from audio to spectrogram v https://librosa.org/doc/main/generated/librosa.load.html
test_graph_audio, sample_rate = librosa.load(test_audio) # sample_rate is the amount of lines that apprear in the spectrogram

plt.plot(test_graph_audio)




In [ ]:
#test with different sample rates
small_graph_audio, sample_rate = librosa.load(test_audio, sr=10000)
large_graph_audio, sample_rate = librosa.load(test_audio, sr=50000)
plt.plot(small_graph_audio)
plt.plot(large_graph_audio)
plt.legend(['sample_rate = 10000', 'sample_rate = 500000'])

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt

y, sr = librosa.load(test_audio)
S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
S_dB = librosa.power_to_db(S, ref=np.max)

plt.figure(figsize=(10, 4))
librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', cmap='coolwarm')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-Frequency Spectrogram')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Count the number of samples for each class
class_counts = data['class'].value_counts()

# Plot the distribution
plt.figure(figsize=(10, 6))
class_counts.plot(kind='bar', color='skyblue')
plt.title('Distribution of Audio Types')
plt.xlabel('Audio Class')
plt.ylabel('Number of Samples')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
from librosa.util import fix_length

def process_audio_cnn(file_path, sample_rate=22050, max_length=128):
    y, sr = librosa.load(file_path, sr=sample_rate)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=128)  # Extract 128 MFCCs
    mfcc = fix_length(mfcc, size=max_length, axis=1)  # Pad or truncate along time axis
    return mfcc


In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
sample_rate = 44100

def load_audio_files(dataframe, helper, sample_rate=sample_rate, duration=4, ): # loads the audio files; can change the sample_rate if necessary but otherwise keep it as default level for normal hertz
    audio_data = []
    labels = []

    with ThreadPoolExecutor() as executor: # Runs the folds in parallel which isn't necessary but pretty useful
        futures = []
        for index, row in dataframe.iterrows():
            # Update file path to match the folder structure shown in the image
            file_path = os.path.join('UrbanSound8K', 'fold' + str(row['fold']), row['slice_file_name'])
            futures.append(executor.submit(helper, file_path, sample_rate))
            labels.append(row['classID'])

        for future in futures:
            audio_data.append(future.result())

    return np.array(audio_data), np.array(labels)


X, y = load_audio_files(data, process_audio_cnn) # X representing the MFCC Features that we extracted and y is representing a number correlated to a list of sounds. 


In [ ]:
print(X.shape, y.shape)


In [ ]:
print(y[1])#labels
print(X[1])#Audio Data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)


In [ ]:
X_train_cnn = X_train.reshape(-1, 128, 128, 1)  
X_test_cnn = X_test.reshape(-1, 128, 128, 1)


In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers



def build_cnn_model(input_shape):
    model = models.Sequential([
    layers.Conv2D(32 , (3,3),activation = 'relu',padding='valid', input_shape = input_shape),  
    layers.MaxPooling2D(2, padding='same'),
    layers.Conv2D(128, (3,3), activation='relu',padding='valid'),
    layers.MaxPooling2D(2, padding='same'),
    layers.Dropout(0.3),
    layers.Conv2D(128, (3,3), activation='relu',padding='valid'),
    layers.MaxPooling2D(2, padding='same'),
    layers.Dropout(0.3),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256 , activation = 'relu'),
    layers.Dense(10 , activation = 'softmax')
])
    opt = tf.keras.optimizers.Adam(learning_rate = 0.001)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model




In [ ]:
print(f"X_train_cnn shape: {X_train_cnn.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test_cnn shape: {X_test_cnn.shape}")
print(f"y_test shape: {y_test.shape}")


In [ ]:
cnn_model = build_cnn_model(input_shape=(128, 128, 1))
cnn_history = cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, validation_data=(X_test_cnn, y_test))


In [ ]:
def process_audio_rnn(file_path, sample_rate): # Also used for LSTM
    audio, sr = librosa.load(file_path, sr=sample_rate)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfccs = np.mean(mfccs.T, axis=0)
    return mfccs

X, y = load_audio_files(data, process_audio_rnn) 


In [ ]:
print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

In [ ]:
X_train_rnn = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_rnn = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

In [ ]:
from tensorflow.keras import models, layers
def build_rnn_model(input_shape):
    model = models.Sequential([
        layers.SimpleRNN(64, input_shape=input_shape, return_sequences=True),
        layers.SimpleRNN(32),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(10, activation='softmax')  # Adjust the output units to match your classification task
        ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:

# Train the RNN Model
input_shape = (X_train_rnn.shape[1], X_train_rnn.shape[2])  # timesteps, features
rnn_model = build_rnn_model(input_shape)
rnn_history = rnn_model.fit(X_train_rnn, y_train, epochs=10, batch_size=32, validation_data=(X_test_rnn, y_test))


In [ ]:
input_shape = (X_train_rnn.shape[1], X_train_rnn.shape[2])  
def build_lstm_model(input_shape):
    model = models.Sequential([
        layers.LSTM(64, input_shape=input_shape, return_sequences=True),
        layers.LSTM(32),
        layers.Dense(64, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
        
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Build and train the LSTM model
input_shape = (X_train_rnn.shape[1], X_train_rnn.shape[2]) 
lstm_model = build_lstm_model(input_shape)
lstm_history = lstm_model.fit(X_train_rnn, y_train, epochs=10, batch_size=32, validation_data=(X_test_rnn, y_test))


In [ ]:
# Evaluate the LSTM model
lstm_eval = lstm_model.evaluate(X_test_rnn, y_test)

# Predict labels for evaluation
y_pred_lstm = np.argmax(lstm_model.predict(X_test_rnn), axis=1)

print("LSTM Accuracy:", lstm_eval[1])

# Classification report
from sklearn.metrics import classification_report
print("LSTM Classification Report:")
print(classification_report(y_test, y_pred_lstm))


In [ ]:
# Plot Training Loss
plt.plot(cnn_history.history['loss'], label='CNN Training Loss')
plt.plot(rnn_history.history['loss'], label='RNN Training Loss')
plt.plot(lstm_history.history['loss'], label='LSTM Training Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Model Training Loss')
plt.show()

In [ ]:
# Validation Loss
plt.plot(cnn_history.history['val_loss'], label='CNN Validation Loss')
plt.plot(rnn_history.history['val_loss'], label='RNN Validation Loss')
plt.plot(lstm_history.history['val_loss'], label='LSTM Validation Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Model Validation Loss')
plt.show()


In [ ]:
# Evaluate and Compare Models
cnn_eval = cnn_model.evaluate(X_test_cnn, y_test)
rnn_eval = rnn_model.evaluate(X_test_rnn, y_test)
lstm_eval = lstm_model.evaluate(X_test_rnn, y_test)

print("CNN Accuracy:", cnn_eval[1])
print("RNN Accuracy:", rnn_eval[1])
print("LSTM Accuracy:", lstm_eval[1])

# Classification Reports
from sklearn.metrics import classification_report

y_pred_cnn = np.argmax(cnn_model.predict(X_test_cnn), axis=1)
y_pred_rnn = np.argmax(rnn_model.predict(X_test_rnn), axis=1)
y_pred_lstm = np.argmax(lstm_model.predict(X_test_rnn), axis=1)

print("CNN Classification Report:")
print(classification_report(y_test, y_pred_cnn))
print("RNN Classification Report:")
print(classification_report(y_test, y_pred_rnn))
print("LSTM Classification Report:")
print(classification_report(y_test, y_pred_lstm))



In [ ]:

# Plot Training History
import matplotlib.pyplot as plt

# Plot accuracy
plt.plot(cnn_history.history['accuracy'], label='CNN Accuracy')
plt.plot(rnn_history.history['accuracy'], label='RNN Accuracy')
plt.plot(lstm_history.history['accuracy'], label='LSTM Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Model Training Accuracy')
plt.show()


In [ ]:
import os
import pandas as pd
import librosa
import numpy as np

# Preprocessing function for CNN input
def preprocess_audio_cnn(file_path, sr=22050, duration=10, n_mels=128, target_size=(128, 128)):
    audio, _ = librosa.load(file_path, sr=sr, duration=duration)
    if len(audio) < sr * duration:
        audio = np.pad(audio, (0, sr * duration - len(audio)))
    else:
        audio = audio[:sr * duration]
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    mel_spec_resized = np.resize(mel_spec, target_size)  
    return mel_spec_resized

# Preprocessing function for RNN/LSTM input
def preprocess_audio_rnn(file_path, sr=22050, duration=4):
    audio, _ = librosa.load(file_path, sr=sr, duration=duration)
    if len(audio) < sr * duration:
        audio = np.pad(audio, (0, sr * duration - len(audio)))
    else:
        audio = audio[:sr * duration]
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfccs = np.mean(mfccs.T, axis=0)  
    return mfccs

# Directory containing test audio files
test_dir = "test"
test_files = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.wav')]

# Preprocess files for CNN
X_test_cnn = []
for file in test_files:
    mel_spec = preprocess_audio_cnn(file)
    mel_spec = np.expand_dims(mel_spec, axis=-1)  # Add channel dimension keep this
    X_test_cnn.append(mel_spec)
X_test_cnn = np.array(X_test_cnn)

# Preprocess files for RNN/LSTM
X_test_rnn = []
for file in test_files:
    mfccs = preprocess_audio_rnn(file)
    X_test_rnn.append(mfccs)
X_test_rnn = np.array(X_test_rnn).reshape(-1, 1, 13)  

# Predictions for CNN, RNN, and LSTM models
cnn_predictions = cnn_model.predict(X_test_cnn)
rnn_predictions = rnn_model.predict(X_test_rnn)
lstm_predictions = lstm_model.predict(X_test_rnn)

# Convert predictions to class labels
cnn_classes = np.argmax(cnn_predictions, axis=1)
rnn_classes = np.argmax(rnn_predictions, axis=1)
lstm_classes = np.argmax(lstm_predictions, axis=1)

# Create a DataFrame to display results
results = pd.DataFrame({
    "File Name": [os.path.basename(f) for f in test_files],
    "CNN Prediction": cnn_classes,
    "RNN Prediction": rnn_classes,
    "LSTM Prediction": lstm_classes
})

# Display the results
print(results)


In [ ]:
# Mapping of file names to their actual labels based on the observed naming convention
file_name_mapping = {
    "aircondition": "Air Conditioner",
    "carhorn": "Car Horn",
    "dogbark": "Dog Bark",
    "drilling": "Drilling",
    "engine": "Engine Idling",
    "gunshot": "Gun Shot",
    "jackhammer": "Jackhammer",
    "siren": "Siren",
    "streetmusic": "Street Music"
}

# Add a column for real answers based on file names
results["Real Answer"] = results["File Name"].apply(
    lambda x: next((label for key, label in file_name_mapping.items() if key in x), "Unknown")
)

# Calculate if predictions are correct
results["CNN Correct"] = results["CNN Prediction"] == results["Real Answer"]
results["RNN Correct"] = results["RNN Prediction"] == results["Real Answer"]
results["LSTM Correct"] = results["LSTM Prediction"] == results["Real Answer"]

# Calculate accuracy for each model
cnn_accuracy = results["CNN Correct"].mean() * 100
rnn_accuracy = results["RNN Correct"].mean() * 100
lstm_accuracy = results["LSTM Correct"].mean() * 100

# Display accuracy results
print(f"CNN Accuracy: {cnn_accuracy:.2f}%")
print(f"RNN Accuracy: {rnn_accuracy:.2f}%")
print(f"LSTM Accuracy: {lstm_accuracy:.2f}%")

# Display updated DataFrame
print(results)


In [ ]:
# Define class mapping
class_mapping = { 
    0: "Air Conditioner",
    1: "Car Horn",
    2: "Children Playing",
    3: "Dog Bark",
    4: "Drilling",
    5: "Engine Idling",
    6: "Gun Shot",
    7: "Jackhammer",
    8: "Siren",
    9: "Street Music"
}

# Map predictions to their corresponding labels
cnn_labels = [class_mapping[pred] for pred in cnn_classes]
rnn_labels = [class_mapping[pred] for pred in rnn_classes]
lstm_labels = [class_mapping[pred] for pred in lstm_classes]

# Create a DataFrame to display results with class names
results = pd.DataFrame({
    "File Name": [os.path.basename(f) for f in test_files],
    "CNN Prediction": cnn_labels,
    "RNN Prediction": rnn_labels,
    "LSTM Prediction": lstm_labels
})

# Display the results
print(results)

